# I-JEPA YOLO26 Downstream Evaluation and Video Analysis

This tutorial transfers the tuned I-JEPA encoder from `best_ssl.pt` into YOLO26, fine-tunes it on labelled football data, evaluates the generated detector, and analyzes the supplied football video.

**Learning goals**

- Transfer the I-JEPA online encoder into YOLO26
- Fine-tune the transferred detector with a configurable labelled-data fraction
- Select and validate the generated best.pt checkpoint
- Evaluate labelled test images comprehensively
- Interpret precision, recall, F1, AP, and mAP
- Run video detection and BoT-SORT tracking
- Analyze confidence, coverage, latency, counts, and tracks
- Export reproducible downstream artifacts


## Configuration

SSL_CHECKPOINT points to the tuned I-JEPA `best_ssl.pt`. The notebook transfers only the online YOLO encoder, creates a detector initialization checkpoint, and then produces a supervised detection `best.pt`.


In [ ]:
SSL_CHECKPOINT = "/kaggle/input/notebooks/rifat963/cse445-ijepa-yolo26-football-ssl-tutorial/ijepa_yolo26_football/best_ssl.pt"
MODEL_NAME = "yolo26n"
EXPERIMENT_NAME = "ijepa_yolo26"
FAST_RUN = False
LABEL_FRACTION = 1.0
TRAIN_EPOCHS = 3 if FAST_RUN else 60
TRAIN_IMAGE_SIZE = 640 if FAST_RUN else 960
TRAIN_BATCH_SIZE = 8 if FAST_RUN else 6
RUN_LABEL_FRACTION_SWEEP = False
LABEL_FRACTIONS = [0.10, 0.25, 0.50, 1.00]
CONFIDENCE = 0.25
IOU = 0.70
IMAGE_SIZE = TRAIN_IMAGE_SIZE
VIDEO_STRIDE = 1
MAX_VIDEO_FRAMES = None


## 1. Configure Kaggle

Select a GPU accelerator and attach the I-JEPA SSL notebook output and football dataset through Add Input. The football dataset also supplies `video.mp4`.


In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps "git+https://github.com/rifat963/ssl-detection-lab.git@main"


In [ ]:
from pathlib import Path
from importlib.metadata import version as installed_version
import json
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import Markdown, Video, display
from PIL import Image
from packaging.version import Version
from ultralytics import YOLO

assert Version(installed_version("ssl-detection-lab")) >= Version("0.8.0")

from ssldet import (
    EvaluationConfig,
    VideoAnalysisConfig,
    analyze_video,
    evaluate,
    transfer_ssl_backbone_to_yolo,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")

assert torch.cuda.is_available(), "Select a GPU accelerator before continuing."
assert 0.0 < LABEL_FRACTION <= 1.0, "LABEL_FRACTION must be in (0, 1]"
SSL_WEIGHTS_FILE = Path(SSL_CHECKPOINT)
assert SSL_WEIGHTS_FILE.is_file(), "Attach the I-JEPA SSL notebook output"

pd.Series({
    "SSL checkpoint": str(SSL_WEIGHTS_FILE),
    "SSL checkpoint MB": round(SSL_WEIGHTS_FILE.stat().st_size / 1024 ** 2, 2),
    "model name": MODEL_NAME,
    "label fraction": LABEL_FRACTION,
    "training preset": "fast" if FAST_RUN else "full",
    "GPU": torch.cuda.get_device_name(0),
})


## 2. Validate the dataset and video


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]
VIDEO_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/video.mp4"),
    Path("/kaggle/input/football-player-detection-yolov8/video.mp4"),
]
DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
VIDEO_FILE = next((path for path in VIDEO_CANDIDATES if path.is_file()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Attach the football-player-detection-yolov8 dataset")
if VIDEO_FILE is None:
    raise FileNotFoundError("video.mp4 was not found in the attached dataset")

SPLITS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

pd.DataFrame([
    {
        "split": split,
        "images": len(image_files(paths["images"])),
        "labels": len(list(paths["labels"].glob("*.txt"))),
    }
    for split, paths in SPLITS.items()
]).set_index("split")


## 3. Create an absolute-path dataset YAML


In [ ]:
source_yamls = sorted(DATASET_ROOT.parent.rglob("data.yaml"))
source_metadata = yaml.safe_load(source_yamls[0].read_text()) if source_yamls else {}
CLASS_NAMES = source_metadata.get("names", ["ball", "goalkeeper", "player", "referee"])
DATA_YAML = Path(f"/kaggle/working/{EXPERIMENT_NAME}_football.yaml")
DATA_YAML.write_text(yaml.safe_dump({
    "train": str(SPLITS["train"]["images"]),
    "val": str(SPLITS["valid"]["images"]),
    "test": str(SPLITS["test"]["images"]),
    "names": CLASS_NAMES,
}, sort_keys=False))
print(DATA_YAML.read_text())


## 4. Transfer the I-JEPA encoder into YOLO26

Only the trained online encoder is transferred. The I-JEPA predictor, target encoder, optimizer, scheduler, and SSL training state are intentionally excluded.


In [ ]:
OUTPUT_ROOT = Path(f"/kaggle/working/{EXPERIMENT_NAME}_downstream")
INITIALIZED_PT = OUTPUT_ROOT / "ijepa_initialized_yolo26n.pt"
transfer_result = transfer_ssl_backbone_to_yolo(
    ssl_checkpoint=SSL_WEIGHTS_FILE,
    output_file=INITIALIZED_PT,
    yolo_model="yolo26n.yaml",
    minimum_coverage=0.95,
)
transfer_report = json.loads(transfer_result.report_json.read_text())
pd.Series({
    "SSL method": transfer_result.ssl_method,
    "source model": transfer_result.source_model,
    "encoder prefix": transfer_result.encoder_prefix,
    "loaded backbone keys": transfer_result.loaded_keys,
    "total backbone keys": transfer_result.total_backbone_keys,
    "coverage": transfer_result.coverage,
    "initialized detector": str(transfer_result.detector_checkpoint),
})


## 5. Fine-tune with labelled football data

`LABEL_FRACTION` affects only the training split. Validation and test images remain complete and unchanged. The default value of 1.0 targets the strongest detector.


In [ ]:
TRAINING_PROJECT = OUTPUT_ROOT / "training"
TRAINING_NAME = f"ijepa_yolo26n_labels_{int(LABEL_FRACTION * 100):03d}pct"
detector = YOLO(str(INITIALIZED_PT))
detector.train(
    data=str(DATA_YAML),
    epochs=TRAIN_EPOCHS,
    imgsz=TRAIN_IMAGE_SIZE,
    batch=TRAIN_BATCH_SIZE,
    workers=2,
    device=0,
    project=str(TRAINING_PROJECT),
    name=TRAINING_NAME,
    exist_ok=True,
    fraction=LABEL_FRACTION,
    optimizer="auto",
    cos_lr=True,
    amp=True,
    patience=max(10, TRAIN_EPOCHS // 4),
    seed=SEED,
    deterministic=False,
    translate=0.10,
    scale=0.40,
    shear=2.0,
    perspective=0.0002,
    flipud=0.0,
    fliplr=0.50,
    mosaic=1.0,
    mixup=0.10,
    close_mosaic=min(10, max(0, TRAIN_EPOCHS // 5)),
    plots=True,
    verbose=True,
)
TRAINING_DIR = TRAINING_PROJECT / TRAINING_NAME
BEST_PT = TRAINING_DIR / "weights" / "best.pt"
LAST_PT = TRAINING_DIR / "weights" / "last.pt"
WEIGHTS_FILE = BEST_PT if BEST_PT.is_file() else LAST_PT
if not WEIGHTS_FILE.is_file():
    raise FileNotFoundError("Ultralytics did not create best.pt or last.pt")
model = YOLO(str(WEIGHTS_FILE))
assert model.task == "detect", f"Expected detection weights, found task={model.task}"
model_names = dict(model.names)
pd.Series({
    "I-JEPA SSL checkpoint": str(SSL_WEIGHTS_FILE),
    "initialized detector": str(INITIALIZED_PT),
    "fine-tuned checkpoint": str(WEIGHTS_FILE),
    "label fraction": LABEL_FRACTION,
    "task": model.task,
    "classes": len(model_names),
    "class names": ", ".join(str(value) for value in model_names.values()),
})


## 6. Review detection training


In [ ]:
RESULTS_CSV = TRAINING_DIR / "results.csv"
if RESULTS_CSV.is_file():
    training_history = pd.read_csv(RESULTS_CSV)
    training_history.columns = [column.strip() for column in training_history.columns]
    display(training_history.tail().round(5))
    metric_columns = [
        column for column in training_history.columns
        if any(token in column.lower() for token in ("map", "precision", "recall", "loss"))
        and pd.api.types.is_numeric_dtype(training_history[column])
    ][:8]
    if metric_columns:
        training_history.plot(
            x="epoch" if "epoch" in training_history.columns else None,
            y=metric_columns,
            subplots=True,
            layout=(len(metric_columns), 1),
            figsize=(12, 3 * len(metric_columns)),
            legend=False,
        )
        plt.tight_layout()
        plt.show()
else:
    print("Training history was not found")


## 7. Inspect sample predictions


In [ ]:
test_images = image_files(SPLITS["test"]["images"])
sample_paths = random.Random(SEED).sample(test_images, min(6, len(test_images)))
sample_results = model.predict(
    source=[str(path) for path in sample_paths],
    imgsz=IMAGE_SIZE,
    conf=CONFIDENCE,
    iou=IOU,
    device=0,
    half=True,
    verbose=False,
)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for axis in axes.flat:
    axis.axis("off")
for axis, result in zip(axes.flat, sample_results):
    axis.imshow(result.plot()[..., ::-1])
    axis.set_title(Path(result.path).name)
plt.tight_layout()
plt.show()


## 8. Evaluate the labelled test split

Evaluation uses a confidence floor of 0.001 so the complete precision-recall curve and AP values can be calculated.


In [ ]:
EVALUATION_DIR = OUTPUT_ROOT / "test_evaluation"
evaluation_result = evaluate(EvaluationConfig(
    model_name=MODEL_NAME,
    weights_file=str(WEIGHTS_FILE),
    data=str(DATA_YAML),
    output_dir=str(EVALUATION_DIR),
    split="test",
    image_size=IMAGE_SIZE,
    batch_size=8,
    confidence=0.001,
    iou=IOU,
    max_detections=300,
    device=0,
    workers=2,
    half=True,
    plots=True,
    save_json=True,
))
evaluation_result


In [ ]:
evaluation_report = json.loads(evaluation_result.metrics_json.read_text())
headline_metrics = evaluation_report["headline_metrics"]
display(pd.Series(headline_metrics, name="test value").to_frame().round(5))
if evaluation_result.per_class_csv is not None:
    display(pd.read_csv(evaluation_result.per_class_csv).round(5))


In [ ]:
evaluation_images = []
for name in ("confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png"):
    matches = sorted(EVALUATION_DIR.rglob(name))
    if matches:
        evaluation_images.append((name, matches[0]))
if evaluation_images:
    fig, axes = plt.subplots(1, len(evaluation_images), figsize=(7 * len(evaluation_images), 6))
    axes = np.atleast_1d(axes)
    for axis, (name, path) in zip(axes, evaluation_images):
        with Image.open(path) as image:
            axis.imshow(image)
        axis.set_title(name.replace("_", " ").replace(".png", ""))
        axis.axis("off")
    plt.tight_layout()
    plt.show()


The labelled test set supports precision, recall, F1, AP, and mAP. Keep it separate from training and model selection.


## 9. Analyze the complete football video


In [ ]:
VIDEO_DIR = OUTPUT_ROOT / "video_analysis"
video_result = analyze_video(VideoAnalysisConfig(
    video_source=str(VIDEO_FILE),
    model_name=MODEL_NAME,
    weights_file=str(WEIGHTS_FILE),
    output_dir=str(VIDEO_DIR),
    confidence=CONFIDENCE,
    iou=IOU,
    image_size=IMAGE_SIZE,
    max_detections=300,
    device=0,
    tracker="botsort.yaml",
    video_stride=VIDEO_STRIDE,
    max_frames=MAX_VIDEO_FRAMES,
    save_annotated=True,
    save_txt=False,
    save_confidence=False,
))
video_result


In [ ]:
video_report = json.loads(video_result.report_json.read_text())
metrics = video_report["video_metrics"]
pd.Series({
    "frames processed": metrics["frames_processed"],
    "total detections": metrics["total_detections"],
    "detection frame coverage": metrics["detection_frame_coverage"],
    "mean confidence": metrics["confidence"]["mean"],
    "mean inference latency ms": metrics["latency_ms"]["mean"],
    "processing FPS": metrics["processing_fps"],
    "unique tracks": metrics["tracking"]["unique_tracks"],
}).to_frame("value")


In [ ]:
per_class = pd.DataFrame(video_report["per_class"])
if not per_class.empty:
    per_class["mean_confidence"] = per_class["confidence"].map(lambda value: value["mean"])
    display(per_class[[
        "class_id",
        "class_name",
        "detections",
        "frames_present",
        "frame_coverage",
        "unique_tracks",
        "mean_confidence",
    ]].round(4))


In [ ]:
frames = pd.read_csv(video_result.frames_csv)
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
sns.lineplot(data=frames, x="frame", y="detections", linewidth=1.3, ax=axes[0])
sns.lineplot(data=frames, x="frame", y="inference_ms", linewidth=1.2, ax=axes[1])
axes[0].set_title("Detections per frame")
axes[1].set_title("Inference latency per frame")
plt.tight_layout()
plt.show()


In [ ]:
display(Markdown(video_result.outcome_markdown.read_text()))


## 10. Display the annotated video


In [ ]:
if video_result.annotated_media:
    display(Video(str(video_result.annotated_media[0]), embed=True, width=960))
else:
    print("No annotated video was discovered")


## 11. Export results


In [ ]:
ARCHIVE = Path(shutil.make_archive(
    f"/kaggle/working/{EXPERIMENT_NAME}_downstream_results",
    "zip",
    root_dir=OUTPUT_ROOT,
))
pd.Series({
    "metrics JSON": str(evaluation_result.metrics_json),
    "video report": str(video_result.report_json),
    "annotated media": ", ".join(str(path) for path in video_result.annotated_media),
    "archive": str(ARCHIVE),
    "archive MB": round(ARCHIVE.stat().st_size / 1024 ** 2, 2),
})


## 12. Optional labelled-data fraction study

Set `RUN_LABEL_FRACTION_SWEEP=True` to compare 10%, 25%, 50%, and 100% of the training labels. Every run starts from the same transferred I-JEPA checkpoint, uses the same seed and training schedule, and is ranked only on the complete validation split. The test split remains untouched until final reporting.


In [ ]:
fraction_results = []
if RUN_LABEL_FRACTION_SWEEP:
    for fraction in LABEL_FRACTIONS:
        run_name = f"ijepa_fraction_{int(fraction * 100):03d}pct"
        if fraction == LABEL_FRACTION:
            fraction_checkpoint = WEIGHTS_FILE
        else:
            fraction_model = YOLO(str(INITIALIZED_PT))
            fraction_model.train(
                data=str(DATA_YAML),
                epochs=TRAIN_EPOCHS,
                imgsz=TRAIN_IMAGE_SIZE,
                batch=TRAIN_BATCH_SIZE,
                workers=2,
                device=0,
                project=str(OUTPUT_ROOT / "label_fraction_sweep"),
                name=run_name,
                exist_ok=True,
                fraction=fraction,
                optimizer="auto",
                cos_lr=True,
                amp=True,
                patience=max(10, TRAIN_EPOCHS // 4),
                seed=SEED,
                deterministic=False,
                translate=0.10,
                scale=0.40,
                shear=2.0,
                perspective=0.0002,
                flipud=0.0,
                fliplr=0.50,
                mosaic=1.0,
                mixup=0.10,
                close_mosaic=min(10, max(0, TRAIN_EPOCHS // 5)),
                plots=False,
                verbose=True,
            )
            fraction_run = OUTPUT_ROOT / "label_fraction_sweep" / run_name
            fraction_checkpoint = fraction_run / "weights" / "best.pt"
        fraction_detector = YOLO(str(fraction_checkpoint))
        validation_metrics = fraction_detector.val(
            data=str(DATA_YAML),
            split="val",
            imgsz=IMAGE_SIZE,
            batch=8,
            device=0,
            half=True,
            plots=False,
            verbose=False,
        )
        fraction_results.append({
            "label fraction": fraction,
            "training images": round(len(image_files(SPLITS["train"]["images"])) * fraction),
            "precision": float(validation_metrics.box.mp),
            "recall": float(validation_metrics.box.mr),
            "mAP50": float(validation_metrics.box.map50),
            "mAP50-95": float(validation_metrics.box.map),
            "checkpoint": str(fraction_checkpoint),
        })
fraction_table = pd.DataFrame(fraction_results)
if fraction_table.empty:
    fraction_table = pd.DataFrame({
        "label fraction": LABEL_FRACTIONS,
        "status": "Set RUN_LABEL_FRACTION_SWEEP=True to run",
    })
display(fraction_table)
if RUN_LABEL_FRACTION_SWEEP and not fraction_table.empty:
    plot_data = fraction_table.melt(
        id_vars="label fraction",
        value_vars=["mAP50", "mAP50-95"],
        var_name="metric",
        value_name="score",
    )
    fig, axis = plt.subplots(figsize=(9, 5))
    sns.lineplot(
        data=plot_data,
        x="label fraction",
        y="score",
        hue="metric",
        marker="o",
        linewidth=2.5,
        ax=axis,
    )
    axis.set_title("I-JEPA label efficiency on the complete validation split")
    axis.set_xticks(LABEL_FRACTIONS)
    axis.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()


## Interpretation boundary

- Precision, recall, F1, AP, mAP, and confusion matrices come from the labelled test split.
- `LABEL_FRACTION` subsamples only training data; validation and test data are never reduced.
- Use validation mAP to compare label fractions and reserve test metrics for the final selected model.
- The 100% label fraction is the default for peak performance; smaller fractions measure I-JEPA data efficiency.
- Video confidence, counts, coverage, latency, and predicted tracks do not measure ground-truth accuracy.
- MOTA, MOTP, IDF1, and HOTA require ground-truth video identities.
- Ultralytics uses AGPL-3.0 or an Enterprise License. Review https://www.ultralytics.com/license.
